# Hallucination Loop — Colab GPU Server

**使い方:**
1. ランタイム → ランタイムのタイプを変更 → **T4 GPU** を選択
2. 全セルを順番に実行（Cell4 まで）
3. Cell4 の出力に表示される `COLAB_SERVER_URL=https://...` をコピー
4. ローカルの `.env` に貼り付けて `python main.py` を実行

**このサーバーが担当する処理:**
- 画像生成: `FLUX.1-schnell` (txt2img / img2img)
- 3D生成: `TripoSR` (image → OBJ mesh)

**ローカルが直接担当する処理:**
- 画像分析: Google Gemini API 無料枠（Colabは使わない）
- Blenderレンダリング: ローカルのまま

In [ ]:
# ── Cell 1: Install Dependencies ─────────────────────────────
import subprocess, sys

packages = [
    "diffusers>=0.31.0",
    "transformers>=4.44.0",
    "accelerate>=0.33.0",
    "sentencepiece",
    "protobuf",
    "fastapi",
    "uvicorn[standard]",
    "python-multipart",
    "Pillow",
    "trimesh",
    "huggingface_hub",
]
subprocess.run([sys.executable, "-m", "pip", "install", "-q"] + packages, check=True)

# TripoSR
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "git+https://github.com/VAST-AI-Research/TripoSR.git"], check=True)

# cloudflared
subprocess.run(
    "wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/"
    "cloudflared-linux-amd64 -O /tmp/cloudflared && chmod +x /tmp/cloudflared",
    shell=True, check=True
)

print('✓ インストール完了')

In [ ]:
# ── Cell 2: Load Models ───────────────────────────────────────
import torch
from diffusers import FluxPipeline, FluxImg2ImgPipeline
from PIL import Image
import io

device = "cuda" if torch.cuda.is_available() else "cpu"
dtype  = torch.bfloat16 if device == "cuda" else torch.float32
gpu_name = torch.cuda.get_device_name(0) if device == "cuda" else "CPU"
print(f"Device: {device} ({gpu_name})")

# ── FLUX.1-schnell ────────────────────────────────────────────
print("Loading FLUX.1-schnell (bfloat16 + CPU offload)...")
txt2img_pipe = FluxPipeline.from_pretrained(
    "black-forest-labs/FLUX.1-schnell",
    torch_dtype=dtype,
)
txt2img_pipe.enable_model_cpu_offload()   # T4 (15GB) に収めるためCPUオフロード使用

img2img_pipe = FluxImg2ImgPipeline.from_pipe(txt2img_pipe)  # コンポーネント共有
print("✓ FLUX.1-schnell 読み込み完了")

# ── TripoSR ───────────────────────────────────────────────────
print("Loading TripoSR...")
from tsr.system import TSR
tsr_model = TSR.from_pretrained(
    "stabilityai/TripoSR",
    config_name="config.yaml",
    weight_name="model.ckpt",
)
tsr_model.renderer.set_chunk_size(131072)
tsr_model = tsr_model.to(device)
print("✓ TripoSR 読み込み完了")

print(f"\nGPU VRAM: {torch.cuda.memory_allocated()/1e9:.1f} GB allocated / "
      f"{torch.cuda.get_device_properties(0).total_memory/1e9:.0f} GB total")

In [ ]:
# ── Cell 3: Define FastAPI Server ─────────────────────────────
from fastapi import FastAPI, UploadFile, File, Form, HTTPException
from fastapi.responses import Response
import traceback

app = FastAPI(title="Hallucination Loop GPU Server")


@app.get("/health")
def health():
    return {"status": "ok", "models": ["FLUX.1-schnell", "TripoSR"], "device": device}


@app.post("/generate_image")
async def generate_image(
    prompt:      str   = Form(...),
    strength:    float = Form(0.6),
    num_steps:   int   = Form(4),
    input_image: UploadFile = File(None),
):
    """txt2img または img2img で PNG を返す。"""
    try:
        if input_image is not None:
            raw = await input_image.read()
            init_img = Image.open(io.BytesIO(raw)).convert("RGB").resize((1024, 1024))
            result = img2img_pipe(
                prompt=prompt,
                image=init_img,
                strength=strength,
                num_inference_steps=num_steps,
                guidance_scale=0.0,
            ).images[0]
        else:
            result = txt2img_pipe(
                prompt=prompt,
                num_inference_steps=num_steps,
                guidance_scale=0.0,
                height=1024,
                width=1024,
            ).images[0]

        buf = io.BytesIO()
        result.save(buf, format="PNG")
        return Response(content=buf.getvalue(), media_type="image/png")

    except Exception as e:
        traceback.print_exc()
        raise HTTPException(status_code=500, detail=str(e))


@app.post("/generate_3d")
async def generate_3d(image: UploadFile = File(...)):
    """TripoSR で image → OBJ メッシュを返す。"""
    try:
        raw = await image.read()
        pil_img = Image.open(io.BytesIO(raw)).convert("RGB")

        with torch.no_grad():
            scene_codes = tsr_model([pil_img], device=device)
            meshes = tsr_model.extract_mesh(scene_codes, resolution=256)

        mesh = meshes[0]
        obj_bytes = mesh.export(file_type="obj")   # trimesh → bytes

        return Response(
            content=obj_bytes,
            media_type="model/obj",
            headers={"Content-Disposition": "attachment; filename=mesh.obj"},
        )

    except Exception as e:
        traceback.print_exc()
        raise HTTPException(status_code=500, detail=str(e))


print("✓ FastAPI アプリ定義完了")

In [ ]:
# ── Cell 4: Launch Server + cloudflared Tunnel ────────────────
import subprocess, threading, re, time
import uvicorn

def _run_server():
    uvicorn.run(app, host="0.0.0.0", port=8000, log_level="warning")

server_thread = threading.Thread(target=_run_server, daemon=True)
server_thread.start()
time.sleep(2)
print("✓ uvicorn 起動")

# cloudflared トンネル起動 → URL 取得
tunnel_proc = subprocess.Popen(
    ["/tmp/cloudflared", "tunnel", "--url", "http://localhost:8000", "--no-autoupdate"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
)

print("トンネル URL を取得中...")
public_url = None
for line in tunnel_proc.stdout:
    text = line.decode("utf-8", errors="replace")
    m = re.search(r"https://[a-z0-9\-]+\.trycloudflare\.com", text)
    if m:
        public_url = m.group(0)
        break

print()
print("=" * 60)
print("✓ サーバー起動完了！")
print()
print(f"  COLAB_SERVER_URL={public_url}")
print()
print("このURLをローカルの .env にコピーしてください。")
print("=" * 60)